# Task 6: Visualise Predictions & Task 8: Documentation
**Marks: 2 + 2 = 4/20** | Pneumonia Detection Assignment


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/pneumonia-detection')
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from src.config import TRANSFER_MODEL_PATH, TEST_DIR
from src.data_loader import load_images_for_viz
from src.visualiser import plot_predictions

In [ ]:
model = tf.keras.models.load_model(TRANSFER_MODEL_PATH)
print('Model loaded:', model.name)
images, true_labels = load_images_for_viz(TEST_DIR, n_per_class=20)
print(f'Loaded {len(images)} test images')

In [ ]:
probs        = model.predict(images, verbose=1).flatten()
pred_labels  = (probs > 0.5).astype(int)
correct      = (pred_labels == true_labels).sum()
print(f'Correct  : {correct} / {len(images)}')
print(f'Incorrect: {len(images) - correct} / {len(images)}')

In [ ]:
plot_predictions(images, true_labels, pred_labels, probs)

In [ ]:
# High confidence correct predictions
class_names   = ['NORMAL', 'PNEUMONIA']
correct_mask  = (pred_labels == true_labels)
confidence    = np.abs(probs - 0.5)
high_conf_idx = np.argsort(confidence * correct_mask)[::-1][:6]

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle('High Confidence Correct Predictions', fontsize=13, fontweight='bold')
for ax, idx in zip(axes, high_conf_idx):
    ax.imshow(images[idx])
    ax.set_title(
        f"True: {class_names[true_labels[idx]]}\n"
        f"Pred: {class_names[pred_labels[idx]]} ({probs[idx]:.2f})",
        color='green', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Hard cases near decision boundary
uncertainty  = np.abs(probs - 0.5)
hard_indices = np.argsort(uncertainty)[:6]

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
fig.suptitle('Hardest Cases - Near Decision Boundary',
             fontsize=13, fontweight='bold')
for ax, idx in zip(axes, hard_indices):
    correct = pred_labels[idx] == true_labels[idx]
    color   = 'green' if correct else 'red'
    ax.imshow(images[idx])
    ax.set_title(
        f"True: {class_names[true_labels[idx]]}\n"
        f"Pred: {class_names[pred_labels[idx]]} ({probs[idx]:.2f})",
        color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Task 6 Observations

**1. What patterns do you notice in the correct predictions?**
Correct predictions show clear distinctions between classes. NORMAL images show clean lung fields without opacities. PNEUMONIA images display prominent consolidations or infiltrates (whitish areas). The model confidently identifies these archetypal cases.

**2. What do the incorrectly predicted images look like visually?**
All 7 errors were False Negatives (PNEUMONIA cases predicted as NORMAL). These images appear borderline with subtle patterns or faint opacities — lying close to the decision boundary, making them challenging even for an advanced model.

**3. What does high confidence on wrong predictions tell us?**
High confidence wrong predictions (e.g., 0.90 probability on a wrong call) indicate the model is very certain about an incorrect classification. This is dangerous as it gives no warning signal. It suggests the model learned non-generalizable features or has slight training data bias.

**4. How should a hospital handle these hard cases?**
- Human-in-the-Loop: All high-confidence pneumonia predictions should be reviewed by a qualified radiologist
- Confidence Thresholds: Require higher confidence (>0.7 or >0.8) before acting on AI diagnosis
- Decision Support Only: AI prediction combined with clinical symptoms and patient history
- Error Analysis: Continuously monitor and retrain on misclassified ambiguous cases


---
# Task 8: Project Summary and Reflection

## How can this model help doctors?
This model serves as a valuable decision support tool:
- Initial Screening: Rapidly screens X-rays to flag potential pneumonia, prioritizing urgent cases and reducing radiologist workload
- Consistency: Provides objective assessment, standardizing diagnostic processes
- Efficiency: Streamlines workflows, letting doctors focus on complex ambiguous cases

## What are its limitations?
- Not a Diagnostic Tool: Human expert must always confirm findings with full clinical picture
- Generalizability: Performance may degrade on X-rays from different hospitals or equipment
- False Negatives: Even 2% missed pneumonia cases can have severe consequences
- Interpretability: CNNs are black boxes — hard to explain why a prediction was made
- Small Validation Set: Only 16 images makes validation metrics unreliable
- Dataset Bias: 74% pneumonia training ratio may bias model towards predicting pneumonia

## Future Improvements
- Larger diverse dataset from multiple hospitals for better generalizability
- Advanced augmentation: CutMix, Mixup, medical-specific techniques
- Explainable AI: Grad-CAM, LIME to visualize model attention regions
- Ensemble methods: Combine multiple CNN architectures
- Lung segmentation: Focus model attention on lung regions only
- Advanced class imbalance handling: Focal loss, cost-sensitive learning
- Clinical integration: Work with medical professionals for workflow integration
- Further regularization: Aggressive dropout, L1/L2 during fine-tuning
